# 02 - Featurisation sanity checks

Spot-checks on ligand graphs, pocket extraction, and ESM-2 embeddings.

Sections needing PDBbind data skip cleanly if `data/raw/` is empty.

In [1]:
from pathlib import Path

import numpy as np

from plb.data import (
    ecfp_fingerprint,
    feature_dims,
    find_default_paths,
    ligand_graph_from_sdf,
    ligand_graph_from_smiles,
    pocket_from_pdb_files,
)
from plb.data.protein import ESMEmbedder, pool_pocket_embedding

DATA_ROOT = Path("../data")
paths = find_default_paths(DATA_ROOT)
have_pdbbind = paths["refined_root"].is_dir()
print(f"PDBbind refined-set found: {have_pdbbind}")
print(f"feature_dims = {feature_dims()}")

PDBbind refined-set found: True
feature_dims = {'node': 36, 'edge': 8}


## 1. Ligand featurisation

Atom/edge counts and ECFP4 bit patterns for a few reference molecules.

In [2]:
examples = {
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "benzene": "c1ccccc1",
    "methane": "C",
}

for name, smiles in examples.items():
    g = ligand_graph_from_smiles(smiles)
    fp = ecfp_fingerprint(smiles)
    print(
        f"{name:>10} | atoms={g.num_atoms:>3} | edges={g.num_edges:>3} | "
        f"ECFP bits set={int(fp.sum()):>3} | smiles={g.smiles}"
    )

   aspirin | atoms= 13 | edges= 26 | ECFP bits set= 24 | smiles=CC(=O)Oc1ccccc1C(=O)O
  caffeine | atoms= 14 | edges= 30 | ECFP bits set= 25 | smiles=Cn1c(=O)c2c(ncn2C)n(C)c1=O
   benzene | atoms=  6 | edges= 12 | ECFP bits set=  3 | smiles=c1ccccc1
   methane | atoms=  1 | edges=  0 | ECFP bits set=  1 | smiles=C


In [3]:
fp_aspirin = ecfp_fingerprint(examples["aspirin"])
fp_benzene = ecfp_fingerprint(examples["benzene"])
fp_caffeine = ecfp_fingerprint(examples["caffeine"])

def tanimoto(a, b):
    intersect = int(np.bitwise_and(a, b).sum())
    union = int(np.bitwise_or(a, b).sum())
    return intersect / max(1, union)

print(f"Tanimoto(aspirin, benzene)  = {tanimoto(fp_aspirin, fp_benzene):.3f}")
print(f"Tanimoto(aspirin, caffeine) = {tanimoto(fp_aspirin, fp_caffeine):.3f}")
print(f"Tanimoto(aspirin, aspirin)  = {tanimoto(fp_aspirin, fp_aspirin):.3f}")

Tanimoto(aspirin, benzene)  = 0.125
Tanimoto(aspirin, caffeine) = 0.089
Tanimoto(aspirin, aspirin)  = 1.000


## 2. Pocket extraction

Runs on real PDBbind complexes if available, otherwise falls back to a synthetic structure.

In [4]:
EXAMPLE_PDB_IDS = ["1a30", "2qbr", "3ptb"]  # adjust to whatever's downloaded

if have_pdbbind:
    refined_root = paths["refined_root"]
    for pdb_id in EXAMPLE_PDB_IDS:
        folder = refined_root / pdb_id
        protein = folder / f"{pdb_id}_protein.pdb"
        ligand = folder / f"{pdb_id}_ligand.sdf"
        if not protein.is_file() or not ligand.is_file():
            print(f"{pdb_id}: not in your refined set, skipping")
            continue
        pocket = pocket_from_pdb_files(protein, ligand, cutoff_angstrom=6.0)
        print(
            f"{pdb_id} | chains={pocket.chain_ids} | total_residues={pocket.n_total_residues} | "
            f"pocket_residues={pocket.n_pocket_residues}"
        )
else:
    print("PDBbind data not found - skipping. Run scripts/download_pdbbind.py first.")

1a30 | chains=['A', 'B'] | total_residues=198 | pocket_residues=28
2qbr | chains=['A'] | total_residues=298 | pocket_residues=30
3ptb: not in your refined set, skipping


[11:16:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [5]:
# Synthetic fallback - works with no real data, mirrors the unit test
import tempfile

SYNTHETIC_PDB = """\
ATOM      1  N   ALA A   1       0.000   0.000   0.000  1.00 20.00           N
ATOM      2  CA  ALA A   1       1.500   0.000   0.000  1.00 20.00           C
ATOM      3  C   ALA A   1       2.000   1.500   0.000  1.00 20.00           C
ATOM      4  O   ALA A   1       3.000   2.000   0.000  1.00 20.00           O
ATOM      5  N   GLY A   2       1.500   2.500   0.000  1.00 20.00           N
ATOM      6  CA  GLY A   2       2.000   4.000   0.000  1.00 20.00           C
ATOM      7  C   GLY A   2       3.500   4.500   0.000  1.00 20.00           C
ATOM      8  O   GLY A   2       4.000   5.500   0.000  1.00 20.00           O
ATOM      9  N   VAL A   3      20.000  20.000  20.000  1.00 20.00           N
ATOM     10  CA  VAL A   3      21.500  20.000  20.000  1.00 20.00           C
ATOM     11  C   VAL A   3      22.000  21.500  20.000  1.00 20.00           C
ATOM     12  O   VAL A   3      23.000  22.000  20.000  1.00 20.00           O
END
"""

SYNTHETIC_SDF = """\
ligand
     RDKit          3D

  1  0  0  0  0  0  0  0  0  0999 V2000
    1.0000    1.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
M  END
$$$$
"""

with tempfile.TemporaryDirectory() as td:
    pdb = Path(td) / "tiny.pdb"
    sdf = Path(td) / "tiny.sdf"
    pdb.write_text(SYNTHETIC_PDB)
    sdf.write_text(SYNTHETIC_SDF)
    pocket = pocket_from_pdb_files(pdb, sdf, cutoff_angstrom=6.0)
    print(f"synthetic | chain={pocket.chain_ids} | seq={pocket.chains}")
    print(f"          | pocket_positions={pocket.pocket_residues}")
    print(f"          | -> {[pocket.chains[c][i] for c in pocket.chain_ids for i in pocket.pocket_residues[c]]}")

synthetic | chain=['A'] | seq={'A': 'AGV'}
          | pocket_positions={'A': [0, 1]}
          | -> ['A', 'G']


## 3. ESM-2 35M embedding

First run downloads weights (~150 MB) to `~/.cache/torch/hub/`. Set `RUN_ESM = False` to skip.

In [6]:
RUN_ESM = True

if RUN_ESM:
    embedder = ESMEmbedder(model_name="esm2_t12_35M_UR50D", device="cpu")
    sequence = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEK"
    emb = embedder.embed_chain(sequence)
    print(f"sequence length: {len(sequence)}")
    print(f"embedding shape: {emb.shape} (expected ({len(sequence)}, {embedder.embed_dim}))")
    print(f"embedding dtype: {emb.dtype}")
    print(f"sample residue 0 (first 8 dims): {emb[0, :8]}")
else:
    print("RUN_ESM=False - skipping ESM model download.")

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to C:\Users\fearg/.cache\torch\hub\checkpoints\esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to C:\Users\fearg/.cache\torch\hub\checkpoints\esm2_t12_35M_UR50D-contact-regression.pt
sequence length: 53
embedding shape: (53, 480) (expected (53, 480))
embedding dtype: float32
sample residue 0 (first 8 dims): [-0.41633883  0.21460612 -0.06959341  0.48728356 -0.02740774  0.26276988
 -0.07364158 -0.17980812]


### Pocket pooling end-to-end

Real complex (or synthetic fallback) -> pocket residues -> ESM -> pooled vector.

In [ ]:
if RUN_ESM:
    if have_pdbbind:
        pdb_id = EXAMPLE_PDB_IDS[0]
        folder = paths["refined_root"] / pdb_id
        protein = folder / f"{pdb_id}_protein.pdb"
        ligand = folder / f"{pdb_id}_ligand.sdf"
        if protein.is_file() and ligand.is_file():
            pocket = pocket_from_pdb_files(protein, ligand)
            print(f"Using real complex {pdb_id}")
        else:
            pocket = None
    else:
        pocket = None

    if pocket is None:
        # Use the synthetic mini-protein from above
        with tempfile.TemporaryDirectory() as td:
            pdb = Path(td) / "tiny.pdb"
            sdf = Path(td) / "tiny.sdf"
            pdb.write_text(SYNTHETIC_PDB)
            sdf.write_text(SYNTHETIC_SDF)
            pocket = pocket_from_pdb_files(pdb, sdf, cutoff_angstrom=6.0)
        print("Using synthetic 3-residue protein")

    chain_embeddings = {cid: embedder.embed_chain(seq) for cid, seq in pocket.chains.items()}
    pocket_vec, whole_vec = pool_pocket_embedding(chain_embeddings, pocket.pocket_residues, embedder.embed_dim)

    print(f"chains: {pocket.chain_ids}")
    print(f"pocket_residues: {pocket.n_pocket_residues} of {pocket.n_total_residues}")
    print(f"pocket_pool: {pocket_vec.shape} {pocket_vec.dtype}")
    print(f"whole_pool : {whole_vec.shape} {whole_vec.dtype}")
    print(f"L2(pocket - whole) = {np.linalg.norm(pocket_vec - whole_vec):.3f}")

Using real complex 1a30
chains: ['A', 'B']
pocket_residues: 28 of 198
pocket_pool: (480,) float32
whole_pool : (480,) float32
L2(pocket - whole) = 2.219


[11:16:59] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


: 

## Next

Run `scripts/precompute_esm.py` over the full refined set (~hours on CPU, ~10 min on GPU), then move on to the XGBoost baseline (notebook 03).